# 06 — Integración de voz y compra

Conectamos audio en español, transcripción revisable, agente con RAG y herramientas, tienda SQLite y respuesta generada con XTTS-v2. Qwen se mantiene en GPU; Whisper, recuperación y síntesis usan CPU en la integración para dejar margen de VRAM.


In [1]:
%pip -q install -r requirements-lab4-modular.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-lab4-modular.txt'
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
from pathlib import Path
from IPython.display import Audio, display
from lab4_asistente import config as C
from lab4_asistente.runtime import cargar_sistema, cargar_modelos_voz


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz


## Seleccionar el perfil de voz

El menú muestra los perfiles disponibles y conserva la identidad seleccionada para el turno de prueba.


In [3]:
import ipywidgets as widgets
from IPython.display import display

VOICES = C.voces_disponibles()
if not VOICES:
    raise FileNotFoundError("No se encontraron grabaciones de voz.")
voice_selector = widgets.Dropdown(options=sorted(VOICES), description="Perfil de voz:")
display(voice_selector)
print("Elige la voz y ejecuta la siguiente celda para cargar la integración.")


Dropdown(description='Perfil de voz:', options=('ana', 'diego', 'elsa', 'estuardo', 'nicole', 'winz'), value='…

Elige la voz y ejecuta la siguiente celda para cargar la integración.


## Cargar la solución integrada

La GPU queda reservada para Qwen. Los modelos de recuperación, reconocimiento y síntesis se ubican en CPU para limitar la ocupación simultánea de VRAM.


In [4]:
COMP = cargar_sistema(cargar_llm=True)
SIS = COMP["sistema"]
VOICE_ID = voice_selector.value
ASR, TTS, cargas_voz = cargar_modelos_voz(VOICE_ID)
CARGAS_INICIALES = {**COMP["cargas"], **cargas_voz}
print("Voz seleccionada:", VOICE_ID)
print("Dispositivo LLM:", COMP["dispositivos"]["llm"])
print("Dispositivos ASR/TTS:", cargas_voz["dispositivo_asr"], "/", cargas_voz["dispositivo_tts"])
print("Cargas iniciales (fuera de la latencia por turno):", CARGAS_INICIALES)


GPU lista: NVIDIA GeForce RTX 5070 Ti Laptop GPU | VRAM libre: 10.8/11.9 GB


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/tts/layers/xtts/xtts_manager.py:5: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.speakers = torch.load(speaker_file_path)
/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/utils/io.py:54: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  return torch.load(f, map_location=map_location, **kwargs)
GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, a

Voz seleccionada: ana
Dispositivo LLM: cuda:0
Dispositivos ASR/TTS: cpu / cpu
Cargas iniciales (fuera de la latencia por turno): {'recuperacion': 5.887722147000204, 'llm': 13.449503085000288, 'asr': 4.19903128399983, 'tts': 17.928695997000432, 'dispositivo_asr': 'cpu', 'dispositivo_tts': 'cpu'}


## Procesar una consulta hablada y permitir corregirla

In [6]:
for ruta in sorted((ROOT / "data").rglob("*.wav")):
    print(ruta.relative_to(ROOT))

data/voice_samples/ana.wav
data/voice_samples/diego.wav
data/voice_samples/elsa.wav
data/voice_samples/estuardo.wav
data/voice_samples/nicole.wav
data/voice_samples/winz.wav


In [9]:
from pathlib import Path
from IPython.display import Audio, display

archivo = "data/voice_samples/nicole.wav"
ruta_audio = Path(archivo).expanduser()

if not ruta_audio.is_absolute():
    ruta_audio = ROOT / ruta_audio

if not ruta_audio.is_file():
    raise FileNotFoundError(f"No se encontró el audio: {ruta_audio}")

print("Transcribiendo:", ruta_audio.name, flush=True)
transcripcion = ASR.transcribir(ruta_audio)
print("Transcripción ASR:", transcripcion)

Transcribiendo: nicole.wav
Transcripción ASR: Hola, esta es mi voz de referencia para el asistente de compras. La grabo en un lugar tranquilo, hablando con calma y a un ritmo natural. Prefiero comprar por voz porque es rápido y cómodo, sobre todo cuando tengo las manos ocupadas en la cocina.


In [10]:
# Déjalo vacío para usar la transcripción.
# Si contiene errores, escribe aquí el texto corregido antes de ejecutar.
correccion = ""

entrada = correccion.strip() or transcripcion.strip()
if not entrada:
    raise ValueError("La consulta está vacía. Revisa el audio o escribe una corrección.")

print("Consulta:", entrada)
print("Procesando con el agente...", flush=True)

respuesta = SIS.agente.turno(entrada)
print("Respuesta:", respuesta["respuesta"])
print(
    "Operaciones:",
    [
        (o["herramienta"], o["resultado"].get("ok"))
        for o in respuesta["operaciones"]
    ],
)

ruta_salida = C.RUTA_OUTPUT / "respuesta_demo.wav"
ruta_salida.parent.mkdir(parents=True, exist_ok=True)

print("Generando audio...", flush=True)
TTS.sintetizar(respuesta["respuesta"], ruta_salida)
display(Audio(filename=str(ruta_salida)))

Consulta: Hola, esta es mi voz de referencia para el asistente de compras. La grabo en un lugar tranquilo, hablando con calma y a un ritmo natural. Prefiero comprar por voz porque es rápido y cómodo, sobre todo cuando tengo las manos ocupadas en la cocina.
Procesando con el agente...
Respuesta: Perfecto, gracias por la información. ¿Podrías decirme qué tipo de producto estás buscando? Por ejemplo, si buscas un teléfono inteligente, un audífono o cualquier otro artículo.
Operaciones: []
Generando audio...


## Qué comprobar en la demostración

La transcripción se revisa antes de llegar al agente. El agente solo accede al catálogo sin etiquetas, consulta precios/stock por herramientas y solicita confirmación después de mostrar la cotización. XTTS usa el perfil `data/voice_profiles/<VOICE_ID>.pt`; la grabación de referencia se resuelve desde el catálogo de voces del repositorio.
